# FACT ORDERS

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Data Reading

In [0]:
df = spark.read.table('medallion_arc_e2e.silver.orders')

display(df)

In [0]:
df.groupBy(df.columns).count().filter(col('count') > 1).count()

In [0]:
df_cus = spark.read.table('medallion_arc_e2e.gold.dim_customers')

df_pro = spark.read.table('medallion_arc_e2e.gold.dim_products')

df_cus.display()

df_pro.display()

In [0]:
from pyspark.sql.functions import col
df_dimcust = spark.read.table('medallion_arc_e2e.gold.dim_customers').select('dim_customer_key', col('customer_id').alias('dim_customer_id'))


df_dimproduct = spark.read.table('medallion_arc_e2e.gold.dim_products').select(col('product_id').alias('dim_product_key'), col('product_id').alias('dim_product_id'))


display(df_dimcust)

df_dimproduct.display()

# Fact Dataframe

In [0]:
df_fact = df.join(df_dimcust, df.customer_id == df_dimcust.dim_customer_id, how = 'left').join(df_dimproduct, df.product_id == df_dimproduct.dim_product_id, 'left')

df_fact_new = df_fact.drop('dim_customer_id', 'dim_product_id', 'customer_id', 'product_id')

display(df_fact_new)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
df_fact_new = df_fact_new.withColumn('rn', row_number().over(Window.partitionBy(col('order_id')).orderBy(col('ingesttime').desc()))).filter(col('rn') == 1).drop('rn')

In [0]:
display(df_fact_new)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('medallion_arc_e2e.gold.fact_orders'):
    dlt_obj = DeltaTable.forName(spark, 'medallion_arc_e2e.gold.fact_orders')

    dlt_obj.alias('trg').merge(df_fact_new.alias('src'), 'trg.order_id = src.order_id AND trg.dim_customer_key = src.dim_customer_key AND trg.dim_product_key = src.dim_product_key')\
                .whenMatchedUpdateAll()\
                .whenNotMatchedInsertAll()\
                .execute()

else:
    df_fact_new.write.mode('overwrite')\
                    .option("path", "abfss://gold-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/fact_orders")\
                    .saveAsTable('medallion_arc_e2e.gold.fact_orders')

In [0]:
df_final = spark.read.table('medallion_arc_e2e.gold.fact_orders')

df_final.count()

In [0]:
%sql
-- TRUNCATE TABLE medallion_arc_e2e.gold.fact_orders;